# SMS Spam Data Exploration - Notebook 1
Alok Chauhan - 251810700318
Aman Kumar - 251810700231
Batch 2C

This is our first notebook. We are trying to load the spam dataset
and clean it before doing any analysis. Sir told us to always check
the data first before jumping into charts.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'pandas'

loading the dataset first, it has 5 columns but sir said
only first 2 are useful so we used usecols

In [ ]:
# loading the file
data = pd.read_csv("spam.csv", encoding="latin-1",
                   usecols=[0, 1], header=0)

data.columns = ["label", "message"]
print(data.shape)
data.head()

NameError: name 'pd' is not defined

okay so the shape is (5572, 2) which means 5572 messages and 2 columns.
let me check what type of values are in label column

In [ ]:
# checking what labels we have
print(data["label"].unique())
print(data["label"].value_counts())

so we have ham and spam. ham means normal messages apparently.
4825 ham and 747 spam in raw data

now checking for missing values because that can cause errors later\

In [ ]:
data.isnull().sum()

good no missing values at all. now checking duplicates

In [ ]:
print("duplicate rows :", data.duplicated().sum())

403 duplicates!! that is a lot. we need to remove them.
probably same spam messages were sent to multiple people
and collected multiple times in the dataset

In [ ]:
# removing duplicates
# data.drop_duplicates()  -- this didnt work first time because
# we forgot to save it back

data = data.drop_duplicates()
data = data.reset_index(drop=True)

print("after removing duplicates:", data.shape)

now 5169 rows. removed 403 duplicates.

cleaning the label column just to be safe
(removing extra spaces, making lowercase)

In [ ]:
data["label"] = data["label"].str.strip().str.lower()

# double checking
print(data["label"].unique())

## Adding new columns

we are adding some extra columns that will help us in analysis later.
like how long is each message, does it have a url, etc.

aman suggested we add a spam_signals column to count
how many suspicious things a message has

In [ ]:
# 1 if spam, 0 if ham - needed for calculations later
data["label_num"] = (data["label"] == "spam").astype(int)

# length stuff
data["char_count"] = data["message"].str.len()
data["word_count"] = data["message"].str.split().str.len()

# does message have these things?
data["has_url"]      = data["message"].str.contains("http|www|.com",
                        case=False, na=False).astype(int)
data["has_number"]   = data["message"].str.contains(r"\d{4,}",
                        na=False).astype(int)
data["has_currency"] = data["message"].str.contains("prize|cash|win|£|$",
                        case=False, na=False).astype(int)
data["has_free"]     = data["message"].str.contains("free",
                        case=False, na=False).astype(int)
data["has_call"]     = data["message"].str.contains("call",
                        case=False, na=False).astype(int)
data["has_txt"]      = data["message"].str.contains("txt|text",
                        case=False, na=False).astype(int)
data["has_phone"]    = data["message"].str.contains(r"\d{10,}",
                        na=False).astype(int)
data["exclamation"]  = data["message"].str.count("!")

# total suspicious signals in one message
data["spam_signals"] = (data["has_url"] + data["has_number"] +
                        data["has_currency"] + data["has_free"] +
                        data["has_call"] + data["has_txt"])

print("done. new columns:")
print(list(data.columns))

lets see some examples of spam vs ham to understand the data better

In [ ]:
print("-- example spam messages --")
print(data[data["label"] == "spam"]["message"].head(3).values)

print("\n-- example ham messages --")
print(data[data["label"] == "ham"]["message"].head(3).values)

okay we can clearly see spam messages are longer and have words
like FREE, CALL, WIN etc. ham is just normal conversation

saving the cleaned file now so alok can use it in notebook 2

In [ ]:
data.to_csv("outputs/spam_cleaned.csv", index=False)
print("saved!")
print(data.shape)

## before and after summary

In [ ]:
summary = pd.DataFrame({
    "step"         : ["original file", "after removing duplicates", "final"],
    "rows"         : [5572, 5169, len(data)],
    "rows removed" : [0, 403, 0]
})
print(summary.to_string(index=False))

making a simple chart to show the before after cleaning and the spam vs ham split

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Data Quality - SMS Spam Dataset", fontsize=13)

# before after bar chart
x     = ["Raw\n(5572)", "No Duplicates\n(5169)", f"Final\n({len(data)})"]
y     = [5572, 5169, len(data)]
color = ["gray", "orange", "green"]
axes[0].bar(x, y, color=color)
axes[0].set_title("Row Count Before vs After")
axes[0].set_ylabel("rows")
for i, v in enumerate(y):
    axes[0].text(i, v + 30, str(v), ha="center")

# spam vs ham
counts = data["label"].value_counts()
axes[1].pie(counts, labels=counts.index, autopct="%1.1f%%",
            colors=["#2ECC71", "#E74C3C"])
axes[1].set_title("Spam vs Ham")

# missing values
miss = pd.DataFrame({"column": ["label", "message"], "missing": [0, 0]})
axes[2].bar(miss["column"], miss["missing"], color="green")
axes[2].set_title("Missing Values (none found)")
axes[2].set_ylabel("count")

plt.tight_layout()
plt.savefig("outputs/01_quality_chart.png", dpi=120)
data.to_csv("outputs/spam_cleaned.csv", index=False)
plt.show()

## findings from notebook 1

- total messages after cleaning: 5169
- spam messages: 653 (12.6%)
- ham messages: 4516 (87.4%)
- no missing values found
- 403 duplicate rows removed

one thing we noticed is that dataset is imbalanced -
way more ham than spam. this is normal for real world data
sir said this is called class imbalance

notebook 2 is done by aman - EDA and charts